# Correlated spiking demo -- pairwise cross-correlation across NDF light levels

Not run against a live database from here -- built on `ra.build_master_mapping_table` /
`ra.compute_ccf` (`src/retinanalysis/utils/correlation_utils.py`).

Covers pairwise cross-correlation (CCF), a 3-cell "triplet" joint-correlation (static 3D
plot), and a population Synchrony-Index-vs-distance analysis. All three reuse the same
cell-mapping-across-NDFs step, `build_master_mapping_table`. Ported from three MATLAB
scripts (pairwise CCF with EI-based cell mapping across NDFs, a triplet 3D-plot extension,
and a population SI-vs-distance analysis).

**Design choices:**
- **Cell mapping across NDFs** reuses `ra.cluster_match()` / `ra.ei_corr()`, the same
  EI-based matching `create_mea_pipeline` already uses.
- **Spike times are the full, un-epoch-split spike train** for each block (not sliced into
  stimulus trials).
- **Which datafile represents each NDF is looked up dynamically** from the database
  (`ra.get_ndf_blocks_for_protocol`, keyed on a protocol name you set below), not a
  hardcoded NDF -> path mapping.
- **Micron conversion:** `AnalysisChunk` computes `microns_per_stixel` dynamically per
  experiment (`microns_per_pixel * canvas_size[0] / numXChecks`, read from real epoch
  parameters in `get_noise_params()`) -- used wherever distances are computed below.
- **Reference/"NDF 0" cross-correlation** is plotted from the classification chunk found by
  `ra.find_classified_noise_chunk` -- this is not necessarily the same recording as the
  NDF-0 row of the correlation protocol's own mapping table, so both are kept separate and
  plotted so you can compare.


In [ ]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Choose an experiment and inspect its blocks

Only `exp_name` needs to be set. This prints every block found for that experiment
(datafile, protocol, NDF) so you can pick, from your *actual* data, which protocol's
recordings should provide the spikes for cross-correlation (`CORRELATION_PROTOCOL_NAME`
below) -- deliberately not guessed/hardcoded, since the MATLAB scripts don't name a single
stimulus this analysis has to run on.

In [ ]:
exp_name = '20251015A'

df_exp_summary = ra.get_exp_summary(exp_name)
if df_exp_summary is not None:
    display(df_exp_summary[['datafile_name', 'protocol_name', 'NDF', 'block_id']].sort_values('NDF'))


## Set the correlation protocol, cell type, and EI-matching threshold

`CORRELATION_PROTOCOL_NAME`: exact `protocol_name` (from the table above) whose recordings
provide the spikes to correlate at each NDF.

`CELL_TYPE`: cell-type string (from the reference chunk's classification file) to
restrict the mapping table to. On/off-typed cells are stored as `"<prefix>/<base>"`,
e.g. `'off/brisk transient'` -- matching is case/separator-insensitive (a space or
hyphen instead of `/` also works), but the real words still have to match. If nothing
matches, the cell below prints the real type strings found in the file so you can copy
one exactly.

`CORR_THRESHOLD`: EI-correlation cutoff for `cluster_match`, passed straight through as
`corr_cutoff`. Default 0.85, matching the MATLAB scripts' threshold.

In [ ]:
CORRELATION_PROTOCOL_NAME = 'manookinlab.protocols.SpatialNoise'  # set to the protocol you want spikes correlated during
CELL_TYPE = 'off/brisk sustained'  # e.g. 'off/brisk transient', 'on/brisk sustained' --
# matched case/separator-insensitively (a plain space or hyphen also works), but this is
# the real stored format (on/off-typed cells are always "<prefix>/<base>"). If this doesn't
# match anything, the cell below prints the real available type strings to copy.
CORR_THRESHOLD = 0.85
GENOTYPE = 'C57 WT'  # free-text label for plot titles only (triplet/SI sections below) -- not used in any analysis

MANUAL_REFERENCE_CHUNK = None  # e.g. 'chunk18' or 'data000' -- set this to skip auto-detection


## Build the master cell-mapping table

Equivalent to the MATLAB scripts' "Create Master Mapping Table" step. Loads the
classification/reference chunk once (`ra.find_classified_noise_chunk`, unless
`MANUAL_REFERENCE_CHUNK` is set), gets every cell of `CELL_TYPE` in it, then for every NDF
found for `CORRELATION_PROTOCOL_NAME` loads that block and EI-maps the reference cells into
it. `d_ndf_blocks` holds the loaded `MEAResponseBlock` for each NDF so the CCF cells below
don't need to reload anything.

In [ ]:
master_table, ref_chunk, d_ndf_blocks = ra.build_master_mapping_table(
    exp_name,
    cell_type=CELL_TYPE,
    protocol_name=CORRELATION_PROTOCOL_NAME,
    reference_chunk_name=MANUAL_REFERENCE_CHUNK,
    corr_threshold=CORR_THRESHOLD,
)

display(master_table)


## Mosaic: every reference cell of `CELL_TYPE`

RF mosaic (receptive field ellipses, one per cell) for every cell of `CELL_TYPE` found in
the reference/classification chunk -- reuses `AnalysisChunk.plot_rfs` (same function the
intro demo and demo 7's mosaic cells use). `label_cells=True` so each ellipse is tagged
with its `Ref_ID`, matching the IDs in `master_table` above.


In [ ]:
ref_chunk.plot_rfs(cell_types=[CELL_TYPE], label_cells=True, b_zoom=True, units='microns')


## Mosaic: cells that matched at every NDF

Same mosaic, restricted to the subset of `master_table['Ref_ID']` with a non-NaN match in
every `NDF{n}_ID` column -- i.e. cells that survived `CORR_THRESHOLD`-level EI matching at
every light level found for `CORRELATION_PROTOCOL_NAME`, not just some of them. This is
also the candidate pool the nearest-neighbor auto-pick below draws from, since only these
cells can contribute a full cross-NDF CCF panel set.


In [ ]:
ndf_cols = [c for c in master_table.columns if c != 'Ref_ID']
passed_all_ndf_mask = master_table[ndf_cols].notna().all(axis=1)
passed_all_ndf_ids = master_table.loc[passed_all_ndf_mask, 'Ref_ID'].astype(int).tolist()

print(f'{len(passed_all_ndf_ids)} / {len(master_table)} cells matched at every NDF '
      f'({ndf_cols}): {passed_all_ndf_ids}')

if len(passed_all_ndf_ids) == 0:
    print('None matched every NDF -- try a lower CORR_THRESHOLD, or check the per-NDF '
          'mapped counts printed while building master_table above. Nothing to plot.')
else:
    ref_chunk.plot_rfs(noise_ids=passed_all_ndf_ids, label_cells=True, b_zoom=True, units='microns')


## Pick two reference cells to correlate

`ra.get_cell_pairwise_distances` computes ordinary Euclidean distance between RF centers
(AnalysisChunk's RF center + per-experiment stixel->micron conversion, same one
`plot_rfs(units='microns')` uses). The cell below auto-picks the closest pair among the
cells that matched every NDF (mosaic above) as `cell_id_A`/`cell_id_B`, and prints every
pair's distance so you can pick a different one instead.

To override: set `cell_id_A`/`cell_id_B` explicitly at the bottom of the next cell (both
must be present in `master_table['Ref_ID']`). A cell only contributes a panel at a given
NDF if it was successfully mapped there (non-NaN in that NDF's column) -- unmapped NDFs are
skipped and printed.


In [ ]:
df_neighbor_pairs = ra.get_cell_pairwise_distances(ref_chunk, passed_all_ndf_ids, units='microns')

if len(df_neighbor_pairs) > 0:
    print(f'{len(df_neighbor_pairs)} pair(s) among the {len(passed_all_ndf_ids)} cells that '
          'matched every NDF, closest first (microns):')
    display(df_neighbor_pairs)
    cell_id_A = int(df_neighbor_pairs.iloc[0]['cell_a'])
    cell_id_B = int(df_neighbor_pairs.iloc[0]['cell_b'])
    print(f'Auto-picked nearest pair: cell_id_A={cell_id_A}, cell_id_B={cell_id_B} '
          f'({df_neighbor_pairs.iloc[0]["distance"]:.1f} um apart).')
else:
    print(f'Fewer than 2 cells matched every NDF (found {len(passed_all_ndf_ids)}) -- cannot '
          'auto-pick a neighbor pair from that pool. Falling back to the first two rows of '
          'master_table (these may not both be mapped at every NDF, so some CCF panels below '
          'may be skipped).')
    cell_id_A = int(master_table['Ref_ID'].iloc[0])
    cell_id_B = int(master_table['Ref_ID'].iloc[1]) if len(master_table) > 1 else None

# MANUAL OVERRIDE -- uncomment and set explicitly to look at a specific pair instead of the
# auto-picked nearest neighbors (any two Ref_ID values from master_table above):
# cell_id_A = ...
# cell_id_B = ...

for label, cid in [('cell_id_A', cell_id_A), ('cell_id_B', cell_id_B)]:
    if cid is not None and cid not in master_table['Ref_ID'].values:
        raise ValueError(f'{label}={cid} is not a Ref_ID in master_table -- pick one from the table above.')


### Neighbor distance cutoff for population analyses

NEW 2026-08-11 (per yas, item 3c): a data-driven upper distance cutoff for which pairs
count as "neighbors" -- used below by the Synchrony Index vs. distance section, so that
population-level plot isn't diluted by pairs of the same cell type that just happen to
sit on opposite sides of the array (which were never going to show meaningful
correlation regardless of distance, and would bias/flatten a real distance
relationship). Computed from each cell's own nearest-neighbor distance
(`ra.compute_nearest_neighbor_distances`) among the cells that matched every NDF -- the
MEDIAN of that per-cell distribution is a robust, per-experiment measurement of how far
apart adjacent cells of this type actually sit (not an assumed/fixed value), and the
cutoff is `NEIGHBOR_DISTANCE_MULTIPLIER` times that median. See
`ra.get_neighbor_distance_cutoff`'s docstring (`correlation_utils.py`) for the full
reasoning. This does NOT affect the auto-picked closest pair above, or the CCF/triplet
sections below -- those already work on one specific chosen pair, not a population.


In [ ]:
NEIGHBOR_DISTANCE_MULTIPLIER = 1.75  # yas: "cap inclusion at ~1.5-2x that median" -- 1.75 splits the range; try 1.5 or 2.0 for either end

neighbor_cutoff_info = ra.get_neighbor_distance_cutoff(
    df_neighbor_pairs, passed_all_ndf_ids, multiplier=NEIGHBOR_DISTANCE_MULTIPLIER,
)

if np.isnan(neighbor_cutoff_info['cutoff']):
    print('Fewer than 2 cells with a defined nearest-neighbor distance -- no cutoff to compute '
          '(Synchrony Index vs. distance below will use every pair, unfiltered).')
else:
    n_within = int((df_neighbor_pairs['distance'] <= neighbor_cutoff_info['cutoff']).sum())
    print(f"Median nearest-neighbor distance: {neighbor_cutoff_info['median_nn_distance']:.1f} um "
          f"(across {len(neighbor_cutoff_info['nn_distances'])} cells)")
    print(f"Neighbor distance cutoff ({NEIGHBOR_DISTANCE_MULTIPLIER}x median): "
          f"{neighbor_cutoff_info['cutoff']:.1f} um")
    print(f"{n_within} / {len(df_neighbor_pairs)} pairs fall within the cutoff -- "
          "only these will be used in the Synchrony Index vs. distance section below.")


## Plot cross-correlations: reference chunk + every mapped NDF

Left-to-right, top-to-bottom: first panel is the classification/reference chunk's own
spike train for `cell_id_A`/`cell_id_B` (matches the MATLAB script's "NDF 0" plot from
`datarun_classify.spikes`), then one panel per NDF row of `master_table` where both cells
were mapped.

In [ ]:
WINDOW_SIZE = 0.05  # seconds, +/- around each reference spike
BIN_SIZE = 0.002  # seconds

panels = []  # list of (title, ccf, bin_centers)

if cell_id_A in ref_chunk.cell_ids and cell_id_B in ref_chunk.cell_ids:
    spikes_a = ra.get_full_spike_times_sec(ref_chunk, cell_id_A)
    spikes_b = ra.get_full_spike_times_sec(ref_chunk, cell_id_B)
    ccf, t = ra.compute_ccf(spikes_a, spikes_b, window_size=WINDOW_SIZE, bin_size=BIN_SIZE)
    panels.append((f'Reference ({ref_chunk.chunk_name}): {cell_id_A} x {cell_id_B}', ccf, t))
else:
    print(f'{cell_id_A} or {cell_id_B} not found in reference chunk {ref_chunk.chunk_name}, skipping reference panel.')

ndf_cols = [c for c in master_table.columns if c != 'Ref_ID']
row_a = master_table.loc[master_table['Ref_ID'] == cell_id_A].iloc[0]
row_b = master_table.loc[master_table['Ref_ID'] == cell_id_B].iloc[0] if cell_id_B is not None else None

for col in ndf_cols:
    ndf_label = col.replace('_ID', '')
    id_a_targ = row_a[col]
    id_b_targ = row_b[col] if row_b is not None else np.nan

    if pd.isna(id_a_targ) or pd.isna(id_b_targ):
        print(f'Skipping {ndf_label}: cell pair not fully mapped (A: {id_a_targ}, B: {id_b_targ}).')
        continue

    resp_block = d_ndf_blocks[float(ndf_label.replace('NDF', ''))]
    id_a_targ, id_b_targ = int(id_a_targ), int(id_b_targ)
    if id_a_targ not in resp_block.cell_ids or id_b_targ not in resp_block.cell_ids:
        print(f'Mapped id(s) {id_a_targ}/{id_b_targ} not found in {ndf_label} spike data, skipping.')
        continue

    spikes_a = ra.get_full_spike_times_sec(resp_block, id_a_targ)
    spikes_b = ra.get_full_spike_times_sec(resp_block, id_b_targ)
    ccf, t = ra.compute_ccf(spikes_a, spikes_b, window_size=WINDOW_SIZE, bin_size=BIN_SIZE)
    panels.append((f'{ndf_label}: {id_a_targ} x {id_b_targ} (mapped from {cell_id_A} x {cell_id_B})', ccf, t))

n_panels = len(panels)
if n_panels == 0:
    print('No panels to plot -- no valid spike data/mapping found for this cell pair.')
else:
    n_cols = min(3, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)
    for i, (title, ccf, t) in enumerate(panels):
        ax = axes[i // n_cols, i % n_cols]
        ax.plot(t, ccf, 'k', linewidth=1.5)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Coincidences')
        ax.grid(True, alpha=0.3)
    for j in range(n_panels, n_rows * n_cols):
        axes[j // n_cols, j % n_cols].axis('off')
    fig.suptitle(f'{exp_name} | {CELL_TYPE} | cells {cell_id_A} x {cell_id_B}', y=1.02)
    fig.tight_layout()


## Triplet: pick a third cell (C)

Auto-picks a third cell close to the existing `cell_id_A`/`cell_id_B` pair, using the same
`df_neighbor_pairs` distance table computed above (closest pairs first): walks the table
and picks the first pair that introduces exactly one cell not already in
`{cell_id_A, cell_id_B}` -- i.e. the closest additional cell to either A or B. Set
`cell_id_C` explicitly below to override.


In [ ]:
cell_id_C = None
if len(df_neighbor_pairs) > 0:
    existing = {cell_id_A, cell_id_B}
    for _, row in df_neighbor_pairs.iterrows():
        a, b = int(row['cell_a']), int(row['cell_b'])
        new_cells = {a, b} - existing
        if len(new_cells) == 1:
            cell_id_C = new_cells.pop()
            print(f'Auto-picked cell_id_C={cell_id_C} (closest additional cell to the '
                  f'existing pair {cell_id_A} x {cell_id_B}).')
            break
    if cell_id_C is None:
        print('Could not auto-pick a third cell (fewer than 3 distinct cells among the '
              'cells that matched every NDF) -- set cell_id_C manually below if you have '
              'another candidate in mind.')
else:
    print('No neighbor-pair table available -- set cell_id_C manually below.')

# MANUAL OVERRIDE -- uncomment and set explicitly to use a specific third cell instead of
# the auto-picked one (any Ref_ID value from master_table above):
# cell_id_C = ...

if cell_id_C is not None and cell_id_C not in master_table['Ref_ID'].values:
    raise ValueError(f'cell_id_C={cell_id_C} is not a Ref_ID in master_table -- pick one from the table above.')


## Triplet joint-correlation (3D)

Static 3D plot only -- rotating video/GIF export is not built. For every spike of
`cell_id_A`, pools the full cross product of every nearby `cell_id_B`/`cell_id_C` relative
spike time (within `WINDOW_SIZE`) into a 2D histogram via `ra.compute_triplet_map` --
reuses the same `WINDOW_SIZE`/`BIN_SIZE` as the pairwise CCF above. One surface for the
reference chunk ("NDF 0"), then one per NDF where all 3 cells are mapped -- an NDF where A,
B, or C didn't map is skipped and printed, same as the pairwise CCF loop.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the '3d' projection

TITLE_FONTSIZE = 12

def plot_triplet_surface(triplet_map, bin_centers, title):
    # X (axis 0 / rows of triplet_map) = cell_id_B relative time, Y (axis 1 / cols) = cell_id_C
    # relative time -- see ra.compute_triplet_map's docstring for why indexing='ij' keeps this
    # mapping consistent from the histogram all the way through to the plot.
    X, Y = np.meshgrid(bin_centers * 1000, bin_centers * 1000, indexing='ij')
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Y, triplet_map, cmap='jet', edgecolor='none', antialiased=True)
    ax.set_zlabel('Coincidences')
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight='bold')
    ax.view_init(elev=30, azim=-45)
    fig.tight_layout()
    return fig

if cell_id_C is None:
    print('cell_id_C not set -- skipping the triplet section entirely.')
else:
    # Reference (NDF 0) panel -- reference chunk's own spikes directly.
    if (cell_id_A in ref_chunk.cell_ids and cell_id_B in ref_chunk.cell_ids
            and cell_id_C in ref_chunk.cell_ids):
        spikes_a = ra.get_full_spike_times_sec(ref_chunk, cell_id_A)
        spikes_b = ra.get_full_spike_times_sec(ref_chunk, cell_id_B)
        spikes_c = ra.get_full_spike_times_sec(ref_chunk, cell_id_C)
        triplet_map, t = ra.compute_triplet_map(spikes_a, spikes_b, spikes_c,
                                                  window_size=WINDOW_SIZE, bin_size=BIN_SIZE)
        plot_triplet_surface(
            triplet_map, t,
            f'{GENOTYPE}, {CELL_TYPE}\nTriplet: {cell_id_A} (Ref) x {cell_id_B} x {cell_id_C}'
        )
    else:
        print(f'{cell_id_A}, {cell_id_B}, or {cell_id_C} not found in reference chunk '
              f'{ref_chunk.chunk_name}, skipping reference triplet.')

    # One panel per NDF where all 3 cells are mapped.
    row_c = master_table.loc[master_table['Ref_ID'] == cell_id_C]
    for col in ndf_cols:
        ndf_label = col.replace('_ID', '')
        id_a_targ = row_a[col]
        id_b_targ = row_b[col] if row_b is not None else np.nan
        id_c_targ = row_c.iloc[0][col] if len(row_c) else np.nan

        if pd.isna(id_a_targ) or pd.isna(id_b_targ) or pd.isna(id_c_targ):
            print(f'Skipping {ndf_label} triplet: not all 3 cells mapped '
                  f'(A: {id_a_targ}, B: {id_b_targ}, C: {id_c_targ}).')
            continue

        resp_block = d_ndf_blocks[float(ndf_label.replace('NDF', ''))]
        id_a_targ, id_b_targ, id_c_targ = int(id_a_targ), int(id_b_targ), int(id_c_targ)
        if (id_a_targ not in resp_block.cell_ids or id_b_targ not in resp_block.cell_ids
                or id_c_targ not in resp_block.cell_ids):
            print(f'Mapped id(s) {id_a_targ}/{id_b_targ}/{id_c_targ} not found in '
                  f'{ndf_label} spike data, skipping.')
            continue

        spikes_a = ra.get_full_spike_times_sec(resp_block, id_a_targ)
        spikes_b = ra.get_full_spike_times_sec(resp_block, id_b_targ)
        spikes_c = ra.get_full_spike_times_sec(resp_block, id_c_targ)
        triplet_map, t = ra.compute_triplet_map(spikes_a, spikes_b, spikes_c,
                                                  window_size=WINDOW_SIZE, bin_size=BIN_SIZE)
        plot_triplet_surface(
            triplet_map, t,
            f'{GENOTYPE}, {CELL_TYPE}\nTriplet: {id_a_targ} (Ref) x {id_b_targ} x {id_c_targ} [{ndf_label}]'
        )


## Synchrony Index vs. distance

`SI = log2(P_joint / P_chance)` for every pair of cells among those that matched every NDF
(`passed_all_ndf_ids`, from the mosaic section above) and fall within the neighbor
distance cutoff computed above, plotted against physical RF-center distance. See
`ra.compute_synchrony_index`'s docstring for the exact formula and two simplifications
relative to a two-stage 1ms-then-downsample binning scheme (both mathematically
identical for a single continuous stimulus recording -- if your real correlation-protocol
block has multiple separate triggers/repeats instead, that needs different handling):
1. Single-step 10ms binning instead of two-stage 1ms-then-OR-downsample-to-10ms.
2. Whole, un-epoch-split spike trains (same convention as the pairwise CCF/triplet sections
   above) instead of per-epoch-trigger stitching.

Distances always come from the REFERENCE chunk's RF centers (`df_neighbor_pairs`, already
computed above) for every panel -- a "fixed map strategy": the same cell pair's physical
distance stays constant across light levels even though the spikes used for SI come from
that NDF's own (differently-sorted) recording.

**UPDATED 2026-08-11 (item 3c):** pairs farther apart than `neighbor_cutoff_info['cutoff']`
(median nearest-neighbor distance x `NEIGHBOR_DISTANCE_MULTIPLIER`, computed in the
"Neighbor distance cutoff" cell above) are dropped before plotting, so distant same-type
pairs that were never going to show real correlation don't dilute the distance
relationship. Each panel's title notes the cutoff actually used.


In [ ]:
BIN_SIZE_SI = 0.01  # seconds (10ms), matches the MATLAB script's bin_size_stats convention

# UPDATED 2026-08-11 (per yas, item 3c): pairs farther apart than the neighbor distance
# cutoff computed above (neighbor_cutoff_info) are dropped before plotting -- avoids the
# population relationship being diluted by same-cell-type pairs that just happen to sit
# on opposite sides of the array. Set NEIGHBOR_DISTANCE_MULTIPLIER = None (or edit that
# cell above) if you want every pair, unfiltered, like the original version of this cell.
_distance_cutoff = neighbor_cutoff_info['cutoff']

def si_vs_distance_panel(label, spike_trains_by_ref_id):
    if len(spike_trains_by_ref_id) < 2:
        print(f'{label}: fewer than 2 cells with spike data, skipping.')
        return None
    df_si = ra.compute_synchrony_index(spike_trains_by_ref_id, bin_size=BIN_SIZE_SI)
    if len(df_si) == 0:
        print(f'{label}: no pairs with a defined Synchrony Index (need shared, nonzero '
              'coincidences).')
        return None
    merged = df_si.merge(df_neighbor_pairs[['cell_a', 'cell_b', 'distance']],
                          on=['cell_a', 'cell_b'], how='inner')
    if len(merged) == 0:
        print(f'{label}: no SI pairs matched a known distance -- unexpected, check '
              'passed_all_ndf_ids.')
        return None

    n_before = len(merged)
    if not np.isnan(_distance_cutoff):
        merged = merged[merged['distance'] <= _distance_cutoff]
    n_dropped = n_before - len(merged)
    if n_dropped > 0:
        print(f'{label}: dropped {n_dropped} / {n_before} pair(s) beyond the '
              f'{_distance_cutoff:.1f} um neighbor distance cutoff.')
    if len(merged) == 0:
        print(f'{label}: no pairs left within the neighbor distance cutoff, skipping.')
        return None

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(merged['distance'], merged['si'], s=15, c='k', alpha=0.3)
    ax.axhline(0, linestyle='--', color='k')
    ax.set_xlim(0, merged['distance'].max() * 1.05)
    ax.set_ylim(-0.5, 2.5)
    ax.set_xlabel('Distance (microns)')
    ax.set_ylabel('Synchrony Index (S)')
    cutoff_note = f' (<= {_distance_cutoff:.0f} um)' if not np.isnan(_distance_cutoff) else ''
    ax.set_title(f'{GENOTYPE} | {CELL_TYPE} | {label}{cutoff_note}')
    ax.grid(True)
    fig.tight_layout()
    return fig

if len(passed_all_ndf_ids) < 2:
    print('Fewer than 2 cells matched every NDF -- nothing to plot for Synchrony Index vs. distance.')
else:
    # Reference / "High Light" panel -- reference chunk's own spikes.
    ref_spike_trains = {rid: ra.get_full_spike_times_sec(ref_chunk, rid) for rid in passed_all_ndf_ids}
    si_vs_distance_panel(f'High Light ({ref_chunk.chunk_name})', ref_spike_trains)

    # One panel per NDF, using that NDF's own mapped target spike trains (keyed back to
    # Ref_ID so they merge with the reference-based distance table above).
    for col in ndf_cols:
        ndf_label = col.replace('_ID', '')
        ndf_val = float(ndf_label.replace('NDF', ''))
        resp_block = d_ndf_blocks[ndf_val]

        ndf_spike_trains = {}
        for rid in passed_all_ndf_ids:
            target_id = master_table.loc[master_table['Ref_ID'] == rid, col].iloc[0]
            if pd.isna(target_id):
                continue
            target_id = int(target_id)
            if target_id in resp_block.cell_ids:
                ndf_spike_trains[rid] = ra.get_full_spike_times_sec(resp_block, target_id)

        si_vs_distance_panel(ndf_label, ndf_spike_trains)
